# 11 - Retrain TCN on REES46 eCommerce Dataset

This notebook retrains the abandonment detection model (TCN) on the **REES46 eCommerce** dataset.



## Imports & Configuration


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os
import sys
import time

sys.path.append(os.path.abspath("../scripts"))
from train_tcn import AbandonmentTCN, ClickstreamDataset

# Configuration
DATA_PATH = "d:/op_ecom/data/oct_df_clean.parquet"
OUTPUT_DIR = "d:/op_ecom/data/processed_rees46"
MODEL_PATH = "d:/op_ecom/tracker/models/tcn_rees46.pth"
ONNX_PATH = "d:/op_ecom/tracker/models/tcn_real_standalone.onnx"

MAX_SEQ_LEN = 20
MAX_SESSIONS = 200_000
NUM_PAGE_TYPES = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EVENT_MAP = {"view": 1, "cart": 2, "purchase": 3}

print(f"Device: {DEVICE}")

## Step 1: Load & Explore Data


In [ ]:
df = pd.read_parquet(DATA_PATH, columns=["event_time", "event_type", "user_id", "user_session"])
print("Total events:", f"{len(df):,}")
print("\nEvent types:")
print(df["event_type"].value_counts())
print("\nUnique sessions:", f"{df['user_session'].nunique():,}")
print("Unique users:", f"{df['user_id'].nunique():,}")
print("\nDate range:", df["event_time"].min(), "to", df["event_time"].max())
df.head()

## Step 2: Build Rigorous Sessions


In [ ]:
df["page_type"] = df["event_type"].map(EVENT_MAP).fillna(1).astype(int)
df = df.sort_values(["user_session", "event_time"])

# Sample sessions
all_session_ids = df["user_session"].unique()
if len(all_session_ids) > MAX_SESSIONS:
    np.random.seed(42)
    sampled_ids = np.random.choice(all_session_ids, MAX_SESSIONS, replace=False)
    df = df[df["user_session"].isin(sampled_ids)]

sessions = df.groupby("user_session")
print("Processing", f"{sessions.ngroups:,}", "sessions...")

X_page_list, X_dur_list, y_list = [], [], []
skipped = 0

for session_id, group in sessions:
    events = group[["page_type", "event_time"]].values
    page_types = [int(e[0]) for e in events]
    timestamps = [e[1] for e in events]
    
    has_purchase = 3 in page_types
    
    if has_purchase:
        first_purchase_idx = page_types.index(3)
        if first_purchase_idx < 1:
            skipped += 1
            continue
        page_types = page_types[:first_purchase_idx]
        timestamps = timestamps[:first_purchase_idx]
        label = 0  # Purchased
    else:
        label = 1  # Abandoned
    
    if len(page_types) < 1:
        skipped += 1
        continue
    
    durs = []
    for k in range(len(page_types)):
        if k < len(page_types) - 1:
            delta = (timestamps[k+1] - timestamps[k])
            if hasattr(delta, "total_seconds"):
                d = delta.total_seconds()
            else:
                d = float(delta) / 1e9
            d = min(max(d, 0), 600)
            durs.append(d / 600.0)
        else:
            durs.append(0.05)
    
    X_page_list.append(page_types)
    X_dur_list.append(durs)
    y_list.append(label)

y_arr = np.array(y_list)
print("\nValid sessions:", f"{len(y_list):,}", f"(skipped {skipped:,})")
print("Abandoned:", f"{y_arr.sum():,.0f}", f"({100*y_arr.mean():.1f}%)")
print("Purchased:", f"{len(y_arr) - y_arr.sum():,.0f}", f"({100*(1-y_arr.mean()):.1f}%)")

## Step 3: Pad & Split Data


In [ ]:
n = len(y_list)
X_page = np.zeros((n, MAX_SEQ_LEN), dtype=np.int64)
X_dur = np.zeros((n, MAX_SEQ_LEN), dtype=np.float32)
y = np.array(y_list, dtype=np.float32)

for i in range(n):
    length = min(len(X_page_list[i]), MAX_SEQ_LEN)
    X_page[i, :length] = X_page_list[i][:length]
    X_dur[i, :length] = X_dur_list[i][:length]

idx = np.arange(n)
idx_train, idx_temp = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42, stratify=y[idx_temp])

os.makedirs(OUTPUT_DIR, exist_ok=True)
for name, split_idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    np.save(os.path.join(OUTPUT_DIR, f"X_page_{name}.npy"), X_page[split_idx])
    np.save(os.path.join(OUTPUT_DIR, f"X_dur_{name}.npy"), X_dur[split_idx])
    np.save(os.path.join(OUTPUT_DIR, f"y_{name}.npy"), y[split_idx])
    print(f"Saved {name}: {len(split_idx):,} samples")

Xp_tr, Xd_tr, y_tr = X_page[idx_train], X_dur[idx_train], y[idx_train]
Xp_vl, Xd_vl, y_vl = X_page[idx_val], X_dur[idx_val], y[idx_val]

## Step 4: Train TCN Model


In [ ]:
train_ds = ClickstreamDataset(Xp_tr, Xd_tr, y_tr)
val_ds = ClickstreamDataset(Xp_vl, Xd_vl, y_vl)
train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4096, shuffle=False)

model = AbandonmentTCN(num_page_types=NUM_PAGE_TYPES).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

best_val_loss = float("inf")
epochs = 15
history = []

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for p, d, label in train_loader:
        p, d, label = p.to(DEVICE), d.to(DEVICE), label.to(DEVICE)
        optimizer.zero_grad()
        out = model(p, d).squeeze()
        loss = criterion(out, label.float())
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for p, d, label in val_loader:
            p, d, label = p.to(DEVICE), d.to(DEVICE), label.to(DEVICE)
            out = model(p, d).squeeze()
            val_loss += criterion(out, label.float()).item()
            preds = (torch.sigmoid(out) > 0.5).float()
            correct += (preds == label).sum().item()
            total += len(label)
    
    avg_tr = train_loss / len(train_loader)
    avg_vl = val_loss / len(val_loader)
    acc = 100 * correct / total
    marker = ""
    
    if avg_vl < best_val_loss:
        best_val_loss = avg_vl
        torch.save(model.state_dict(), MODEL_PATH)
        marker = " <-- BEST"
    
    history.append({"epoch": epoch+1, "train_loss": avg_tr, "val_loss": avg_vl, "accuracy": acc})
    print(f"Epoch {epoch+1:2d}/{epochs} | Train: {avg_tr:.4f} | Val: {avg_vl:.4f} | Acc: {acc:.1f}%{marker}")

print(f"\nBest Val Loss: {best_val_loss:.4f}")

## Step 5: Training Curves


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_list = [h["epoch"] for h in history]
ax1.plot(epochs_list, [h["train_loss"] for h in history], label="Train Loss")
ax1.plot(epochs_list, [h["val_loss"] for h in history], label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_list, [h["accuracy"] for h in history], color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Validation Accuracy")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Export to ONNX


In [ ]:
model_export = AbandonmentTCN(num_page_types=NUM_PAGE_TYPES)
model_export.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model_export.eval()

dummy_pages = torch.zeros(1, MAX_SEQ_LEN, dtype=torch.long)
dummy_durs = torch.zeros(1, MAX_SEQ_LEN, dtype=torch.float32)

torch.onnx.export(
    model_export,
    (dummy_pages, dummy_durs),
    ONNX_PATH,
    input_names=["page_ids", "durations"],
    output_names=["logits"],
    dynamic_axes={"page_ids": {0: "batch"}, "durations": {0: "batch"}},
    opset_version=17
)

print("Exported:", ONNX_PATH)
print("File size:", f"{os.path.getsize(ONNX_PATH) / 1024:.0f} KB")

## Step 7: Verify ONNX Model


In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(ONNX_PATH)
print("Inputs:", [(i.name, i.shape, i.type) for i in sess.get_inputs()])
print("Outputs:", [(o.name, o.shape, o.type) for o in sess.get_outputs()])

def test_pattern(label, page_arr, dur_arr):
    r = sess.run(None, {"page_ids": page_arr, "durations": dur_arr})
    prob = 1 / (1 + np.exp(-r[0][0][0]))
    print(f"  {label:40s} => {prob*100:.1f}% risk")

print("\n--- Verification Tests ---")

# All zeros
test_pattern("All zeros (padding)", np.zeros((1,20), np.int64), np.zeros((1,20), np.float32))

# Single view
p = np.zeros((1,20), np.int64); d = np.zeros((1,20), np.float32)
p[0,0] = 1; d[0,0] = 0.05
test_pattern("Single view + 19 padding", p, d)

# Browse + cart (padded)
p = np.zeros((1,20), np.int64); d = np.zeros((1,20), np.float32)
p[0,:5] = [1,1,2,1,1]; d[0,:5] = [0.1,0.05,0.02,0.15,0.05]
test_pattern("Browse+cart + 15 padding", p, d)

# View-only (filled like tracker.js)
raw = [(1,0.1),(1,0.08),(1,0.12)]
p = np.zeros((1,20), np.int64); d = np.zeros((1,20), np.float32)
for i in range(20):
    s = raw[i % len(raw)]
    p[0,i] = s[0]; d[0,i] = s[1] if i < 19 else 0.05
test_pattern("View-only pattern (filled 20)", p, d)

# Cart pattern (filled)
raw = [(1,0.1),(1,0.05),(2,0.02),(1,0.15)]
p = np.zeros((1,20), np.int64); d = np.zeros((1,20), np.float32)
for i in range(20):
    s = raw[i % len(raw)]
    p[0,i] = s[0]; d[0,i] = s[1] if i < 19 else 0.05
test_pattern("Cart pattern (filled 20)", p, d)

# Heavy cart (filled)
raw = [(1,0.1),(2,0.05),(1,0.02),(2,0.15),(1,0.08),(2,0.03)]
p = np.zeros((1,20), np.int64); d = np.zeros((1,20), np.float32)
for i in range(20):
    s = raw[i % len(raw)]
    p[0,i] = s[0]; d[0,i] = s[1] if i < 19 else 0.05
test_pattern("Heavy cart (filled 20)", p, d)

print("\n[DONE] Model exported and verified.")